# Person 2 — Tuned T-2h Models (Folds 1–2 Tuning, Folds 3–4 Evaluation)
Run each cell in order from top to bottom.

In [ ]:
# Cell 1: Install dependencies & write project files into Colab workspace
!pip install -q requests pandas pyarrow airportsdata numpy scikit-learn joblib

from pathlib import Path
import json

FILES = {'requirements.txt': 'requests==2.34.2\npandas==3.0.6\npyarrow==26.0.1\nairportsdata==20261123\nnumpy==2.5.3\nscikit-learn==1.9.1\n', 'preprocessing.py': '"""Common T-2h features, chronological folds and unfitted transformers."""\n\nfrom pathlib import Path\n\nimport pandas as pd\nfrom sklearn.base import BaseEstimator, TransformerMixin, clone\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.decomposition import PCA\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler\n\nDATA_DIR = Path(__file__).resolve().parent / "final_csv"\nTARGET = "LATE"\nMETADATA = [\n    "row_id",\n    "FlightDate",\n    "scheduled_dep_utc",\n    "checkpoint_utc",\n    "wx_obs_time",\n]\nSCHEDULE = [\n    "month",\n    "dow",\n    "dep_hour",\n    "arr_hour",\n    "CRSElapsedTime",\n    "Distance",\n    "is_holiday",\n]\nROUTE = ["Origin", "Dest", "Reporting_Airline"]\nWEATHER = [\n    "wx_skyc1_T2h",\n    "wx_tmpf_T2h",\n    "wx_dwpf_T2h",\n    "wx_drct_T2h",\n    "wx_sknt_T2h",\n    "wx_gust_T2h",\n    "wx_vsby_T2h",\n    "wx_p01i_T2h",\n    "wx_skyl1_T2h",\n    "wx_obs_age_min_T2h",\n    "wx_ts_T2h",\n    "wx_fg_T2h",\n    "wx_ra_T2h",\n    "wx_sn_T2h",\n    "wx_fz_T2h",\n    "wx_br_hz_T2h",\n    "wx_gust_missing_T2h",\n    "wx_skyl1_missing_T2h",\n    "codes_known_T2h",\n    "wx_unmatched_T2h",\n]\nAIRPORT_ACTIVITY = [\n    "n_departed_60min_T2h",\n    "prior_mean_delay_60min_T2h",\n    "late_rate_60min_T2h",\n    "n_scheduled_next_60min_T2h",\n    "has_history_T2h",\n]\nFEATURES = SCHEDULE + ROUTE + WEATHER + AIRPORT_ACTIVITY\nCATEGORICAL = ["Origin", "Dest", "Reporting_Airline", "wx_skyc1_T2h"]\nFOLDS = [\n    (1, "2025-09-01", "2025-09-21"),\n    (2, "2025-09-22", "2025-10-12"),\n    (3, "2025-10-13", "2025-11-02"),\n    (4, "2025-11-03", "2025-11-28"),\n]\nGAP_DAYS = 1\nassert len(FEATURES) == len(set(FEATURES)) == 35\n\n\ndef load_development(data_dir=DATA_DIR):\n    """Read and verify the development CSVs; the test CSV is outside this function."""\n    parts = []\n    for name, first, last in (\n        ("train", "2025-01-01", "2025-08-30"),\n        ("validation", "2025-09-01", "2025-11-28"),\n    ):\n        frame = pd.read_csv(\n            Path(data_dir) / f"{name}_T2h.csv", parse_dates=METADATA[2:]\n        )\n        if list(frame) != METADATA + FEATURES + [TARGET]:\n            raise ValueError(f"Unexpected columns in {name}_T2h.csv")\n        if not frame.FlightDate.between(first, last).all():\n            raise ValueError(f"Unexpected dates in {name}_T2h.csv")\n        parts.append(frame)\n    dev = pd.concat(parts, ignore_index=True).set_index("row_id")\n    if not dev.index.is_unique or not dev[TARGET].isin((0, 1)).all():\n        raise ValueError("Duplicate row IDs or invalid target")\n    if not (dev.scheduled_dep_utc - dev.checkpoint_utc).eq(pd.Timedelta(hours=2)).all():\n        raise ValueError("A checkpoint is not two hours before departure")\n    age = (dev.checkpoint_utc - dev.wx_obs_time).dt.total_seconds() / 60\n    if not age.dropna().between(0, 180).all():\n        raise ValueError("Invalid weather observation time")\n    return dev.sort_values("checkpoint_utc", kind="stable")\n\n\ndef development_folds(dev):\n    """Expanding folds with one full local-date gap before each validation block."""\n    for fold, first, last in FOLDS:\n        train = dev[\n            pd.to_datetime(dev.FlightDate)\n            < pd.Timestamp(first) - pd.Timedelta(days=GAP_DAYS)\n        ]\n        validation = dev[dev.FlightDate.between(first, last)]\n        if train.empty or validation.empty:\n            raise ValueError(f"Fold {fold} has no rows")\n        if train.checkpoint_utc.max() >= validation.checkpoint_utc.min():\n            raise ValueError(f"Fold {fold} overlaps in checkpoint time")\n        yield fold, train, validation\n\n\nclass TopDestinations(BaseEstimator, TransformerMixin):\n    """Group destinations outside the training fold\'s most frequent 20."""\n\n    def __init__(self, count=20):\n        self.count = count\n\n    def fit(self, values, y=None):\n        self.keep_ = set(\n            pd.DataFrame(values).iloc[:, 0].value_counts().head(self.count).index\n        )\n        return self\n\n    def transform(self, values):\n        column = pd.DataFrame(values).iloc[:, 0]\n        return column.where(column.isin(self.keep_), "Other").to_numpy().reshape(-1, 1)\n\n\nclass EmptyColumnGuard(TransformerMixin, BaseEstimator):\n    """Represent numeric inputs entirely missing during fit by a fixed constant."""\n\n    def fit(self, X, y=None):\n        self.empty_ = X.columns[X.isna().all()]\n        return self\n\n    def transform(self, X):\n        return X.assign(**{col: 0.0 for col in self.empty_})\n\n\ndef make_preprocessor(kind, features=FEATURES):\n    """Return an unfitted transformer: onehot, onehot_scaled, or ordinal."""\n    if (\n        not features\n        or len(set(features)) != len(features)\n        or set(features) - set(FEATURES)\n    ):\n        raise ValueError("Select distinct approved T-2h inputs")\n    cats = [name for name in features if name in CATEGORICAL]\n    nums = [name for name in features if name not in CATEGORICAL]\n    fill = ("missing", SimpleImputer(strategy="constant", fill_value="__MISSING__"))\n    if kind == "ordinal":\n        parts = []\n        if cats:\n            parts.append(\n                (\n                    "categorical",\n                    Pipeline(\n                        [\n                            fill,\n                            (\n                                "encode",\n                                OrdinalEncoder(\n                                    handle_unknown="use_encoded_value",\n                                    unknown_value=-1,\n                                    max_categories=254,\n                                ),\n                            ),\n                        ]\n                    ),\n                    cats,\n                )\n            )\n        if nums:\n            parts.append(("numeric", EmptyColumnGuard(), nums))\n        return ColumnTransformer(parts, sparse_threshold=0)\n    if kind not in ("onehot", "onehot_scaled"):\n        raise ValueError("kind must be onehot, onehot_scaled, or ordinal")\n    parts = []\n    if "Dest" in cats:\n        parts.append(\n            (\n                "destination",\n                Pipeline(\n                    [\n                        fill,\n                        ("top20", TopDestinations()),\n                        (\n                            "encode",\n                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),\n                        ),\n                    ]\n                ),\n                ["Dest"],\n            )\n        )\n    other = [name for name in cats if name != "Dest"]\n    if other:\n        parts.append(\n            (\n                "categorical",\n                Pipeline(\n                    [\n                        fill,\n                        (\n                            "encode",\n                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),\n                        ),\n                    ]\n                ),\n                other,\n            )\n        )\n    if nums:\n        numeric_steps = [\n            ("impute", SimpleImputer(strategy="median", keep_empty_features=True))\n        ]\n        if kind == "onehot_scaled":\n            numeric_steps.append(("scale", StandardScaler()))\n        parts.append(("numeric", Pipeline(numeric_steps), nums))\n    return ColumnTransformer(parts, sparse_threshold=0)\n\n\ndef categorical_frame(frame, features=FEATURES):\n    """Copy selected inputs with string categories for native categorical APIs."""\n    if (\n        not features\n        or len(set(features)) != len(features)\n        or set(features) - set(FEATURES)\n    ):\n        raise ValueError("Select distinct approved T-2h inputs")\n    result = frame.loc[:, features].copy()\n    for name in CATEGORICAL:\n        if name in result:\n            result[name] = (\n                result[name].astype("string").fillna("__MISSING__").astype("category")\n            )\n    return result\n\n\n# Only the representation and input requirements live here. The caller supplies\n# the estimator, hyperparameters and seed.\nENCODING = {\n    "decision_tree": "onehot",\n    "random_forest": "onehot",\n    "adaboost": "onehot",\n    "gradient_boosting": "onehot",\n    "logistic_regression": "onehot_scaled",\n    "pca_logistic_regression": "onehot_scaled",\n    "histgb": "ordinal",\n    "xgboost": "category",\n    "lightgbm": "category",\n    "catboost": "string",\n}\n\n\nclass NativeInputs(TransformerMixin, BaseEstimator):\n    """Preserve column names and learn a consistent category vocabulary per fold."""\n\n    def __init__(self, kind):\n        self.kind = kind\n\n    def fit(self, X, y=None):\n        self.levels_ = {c: list(X[c].dropna().unique()) for c in CATEGORICAL}\n        return self\n\n    def transform(self, X):\n        frame = X[FEATURES].copy()\n        for col, levels in self.levels_.items():\n            if self.kind == "string":\n                frame[col] = (\n                    frame[col].astype("string").fillna("__MISSING__").astype(object)\n                )\n            else:\n                frame[col] = pd.Categorical(\n                    frame[col].where(frame[col].isin(levels)), categories=levels\n                )\n        return frame\n\n\ndef model_pipeline(model_name, estimator):\n    """Wrap a teammate\'s unfitted estimator with its agreed T-2h preprocessing."""\n    if model_name not in ENCODING:\n        raise ValueError(f"Choose a model key from {list(ENCODING)}")\n    kind = ENCODING[model_name]\n    prep = (\n        NativeInputs(kind)\n        if kind in ("category", "string")\n        else make_preprocessor(kind)\n    )\n    model = clone(estimator)\n    if model_name == "histgb":\n        model.set_params(\n            categorical_features=list(range(len(CATEGORICAL))), early_stopping=False\n        )\n    elif model_name == "xgboost":\n        model.set_params(enable_categorical=True)\n    elif model_name == "catboost":\n        model.set_params(cat_features=list(CATEGORICAL))\n    steps = [("prep", prep)]\n    if model_name == "pca_logistic_regression":\n        steps.append(("pca", PCA(n_components=0.95, svd_solver="full")))\n    return Pipeline([*steps, ("model", model)])\n\n\ndef chronological_cv(dev):\n    """Positional fold indices for sklearn search and cross-validation on dev in this order."""\n    return [\n        (dev.index.get_indexer(train.index), dev.index.get_indexer(val.index))\n        for _, train, val in development_folds(dev)\n    ]\n\n\nif __name__ == "__main__":\n    development = load_development()\n    for fold, train, validation in development_folds(development):\n        print(\n            f"Fold {fold}: {len(train):,} train, {len(validation):,} validation, "\n            f"{validation.FlightDate.min()} to {validation.FlightDate.max()}"\n        )\n', 'decision_tree.py': '"""Tune a T-2h decision tree on folds 1-2, evaluate on 3-4 and save a development refit."""\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import ParameterGrid\nfrom sklearn.tree import DecisionTreeClassifier\n\nfrom preprocessing import (\n    FEATURES,\n    TARGET,\n    development_folds,\n    load_development,\n    model_pipeline,\n)\n\nSEEDS = (11, 23, 42)\nGRID = {\n    "max_depth": [8, 12, 16],\n    "min_samples_leaf": [150, 500],\n    "ccp_alpha": [0.0, 1e-5],\n}\n\n\ndef make_tree(seed, params):\n    return DecisionTreeClassifier(random_state=seed, **params)\n\n\ndef run(output):\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(f"Choose an empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    dev = load_development()\n    tuning, evaluation = [], []\n    best = None\n    for fold, train, val in development_folds(dev):\n        # All candidates use the same fold-local matrix. Float32 is the tree\'s input dtype.\n        prep = model_pipeline("decision_tree", make_tree(SEEDS[0], {}))["prep"]\n        X = prep.fit_transform(train[FEATURES]).astype(np.float32)\n        V = prep.transform(val[FEATURES]).astype(np.float32)\n        y, target = train[TARGET], val[TARGET]\n        if fold <= 2:\n            configs, seeds = ParameterGrid(GRID), SEEDS[:1]\n        else:\n            if best is None:\n                raise ValueError("The first two tuning folds must precede evaluation.")\n            configs, seeds = [best], SEEDS\n        for params in configs:\n            for seed in seeds:\n                start = perf_counter()\n                tree = make_tree(seed, params).fit(X, y)\n                prob = tree.predict_proba(V)[:, 1]\n                row = dict(\n                    fold=fold,\n                    seed=seed,\n                    **params,\n                    train_rows=len(y),\n                    validation_rows=len(target),\n                    late_rate=target.mean(),\n                    ap=average_precision_score(target, prob),\n                    roc_auc=roc_auc_score(target, prob),\n                    seconds=perf_counter() - start,\n                )\n                (tuning if fold <= 2 else evaluation).append(row)\n                if fold > 2:\n                    val[[TARGET]].assign(probability=prob).to_parquet(\n                        output / f"predictions_{fold}_{seed}.parquet"\n                    )\n        print(\n            f"Finished fold {fold}: {len(y):,} training, {len(target):,} validation.",\n            flush=True,\n        )\n        if fold == 2:\n            table = pd.DataFrame(tuning)\n            table.to_csv(output / "tuning.csv", index=False)\n            # Prefer higher AP, then ROC-AUC, then shallower depth in a tie.\n            ranking = table.groupby(list(GRID), as_index=False)[\n                ["ap", "roc_auc"]\n            ].mean()\n            winner = ranking.sort_values(\n                ["ap", "roc_auc", "max_depth"], ascending=[False, False, True]\n            ).iloc[0]\n            best = {\n                "max_depth": int(winner.max_depth),\n                "min_samples_leaf": int(winner.min_samples_leaf),\n                "ccp_alpha": float(winner.ccp_alpha),\n            }\n            print("Selected", best, flush=True)\n        del X, V\n    results = pd.DataFrame(evaluation)\n    results.to_csv(output / "evaluation.csv", index=False)\n    means = results.groupby("fold")[["ap", "roc_auc", "late_rate"]].mean()\n    print(means.to_string(), flush=True)\n    print("Mean across evaluation folds", means.mean().to_dict(), flush=True)\n    final = model_pipeline("decision_tree", make_tree(SEEDS[0], best)).fit(\n        dev[FEATURES], dev[TARGET]\n    )\n    joblib.dump(final, output / "decision_tree.joblib", compress=3)\n    (output / "settings.json").write_text(\n        json.dumps(\n            {\n                "params": best,\n                "refit_seed": SEEDS[0],\n                "features": FEATURES,\n                "development_rows": len(dev),\n                "mean_validation": means.mean().to_dict(),\n            },\n            indent=2,\n        )\n        + "\\n"\n    )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, default=Path("runs/decision_tree"))\n    run(parser.parse_args().out)\n', 'random_forest.py': '"""Tune a T-2h random forest on folds 1-2, evaluate on 3-4 and save a development refit."""\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import ParameterGrid\n\nfrom preprocessing import (\n    FEATURES,\n    TARGET,\n    development_folds,\n    load_development,\n    model_pipeline,\n)\n\nSEEDS = (11, 23, 42)\nGRID = {\n    "n_estimators": [100, 200],\n    "max_depth": [12, 16],\n    "min_samples_leaf": [12, 50],\n}\n\n\ndef make_forest(seed, params):\n    return RandomForestClassifier(random_state=seed, n_jobs=-1, **params)\n\n\ndef run(output):\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(f"Choose an empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    dev = load_development()\n    tuning, evaluation = [], []\n    best = None\n    for fold, train, val in development_folds(dev):\n        prep = model_pipeline("random_forest", make_forest(SEEDS[0], {}))[0]\n        X = prep.fit_transform(train[FEATURES]).astype(np.float32)\n        V = prep.transform(val[FEATURES]).astype(np.float32)\n        y, target = train[TARGET], val[TARGET]\n        if fold <= 2:\n            configs, seeds = ParameterGrid(GRID), SEEDS[:1]\n        else:\n            if best is None:\n                raise ValueError("The first two tuning folds must precede evaluation.")\n            configs, seeds = [best], SEEDS\n        for params in configs:\n            for seed in seeds:\n                start = perf_counter()\n                model = make_forest(seed, params).fit(X, y)\n                prob = model.predict_proba(V)[:, 1]\n                row = dict(\n                    fold=fold,\n                    seed=seed,\n                    **params,\n                    train_rows=len(y),\n                    validation_rows=len(target),\n                    late_rate=target.mean(),\n                    ap=average_precision_score(target, prob),\n                    roc_auc=roc_auc_score(target, prob),\n                    seconds=perf_counter() - start,\n                )\n                (tuning if fold <= 2 else evaluation).append(row)\n                if fold > 2:\n                    val[[TARGET]].assign(probability=prob).to_parquet(\n                        output / f"predictions_{fold}_{seed}.parquet"\n                    )\n        print(\n            f"Finished fold {fold}: {len(y):,} training, {len(target):,} validation.",\n            flush=True,\n        )\n        if fold == 2:\n            table = pd.DataFrame(tuning)\n            table.to_csv(output / "tuning.csv", index=False)\n            ranking = table.groupby(list(GRID), as_index=False)[\n                ["ap", "roc_auc"]\n            ].mean()\n            winner = ranking.sort_values(\n                ["ap", "roc_auc", "n_estimators"], ascending=[False, False, True]\n            ).iloc[0]\n            best = {\n                "n_estimators": int(winner.n_estimators),\n                "max_depth": int(winner.max_depth),\n                "min_samples_leaf": int(winner.min_samples_leaf),\n            }\n            print("Selected", best, flush=True)\n        del X, V\n    results = pd.DataFrame(evaluation)\n    results.to_csv(output / "evaluation.csv", index=False)\n    means = results.groupby("fold")[["ap", "roc_auc", "late_rate"]].mean()\n    print(means.to_string(), flush=True)\n    print("Mean across evaluation folds", means.mean().to_dict(), flush=True)\n    final = model_pipeline("random_forest", make_forest(SEEDS[0], best)).fit(\n        dev[FEATURES], dev[TARGET]\n    )\n    joblib.dump(final, output / "random_forest.joblib", compress=3)\n    (output / "settings.json").write_text(\n        json.dumps(\n            {\n                "params": best,\n                "refit_seed": SEEDS[0],\n                "features": FEATURES,\n                "development_rows": len(dev),\n                "mean_validation": means.mean().to_dict(),\n            },\n            indent=2,\n        )\n        + "\\n"\n    )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, default=Path("runs/random_forest"))\n    run(parser.parse_args().out)\n', 'adaboost.py': '"""Tune a T-2h AdaBoost ensemble on folds 1-2, evaluate on 3-4 and save a development refit."""\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import AdaBoostClassifier\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import ParameterGrid\nfrom sklearn.tree import DecisionTreeClassifier\n\nfrom preprocessing import (\n    FEATURES,\n    TARGET,\n    development_folds,\n    load_development,\n    model_pipeline,\n)\n\nSEEDS = (11, 23, 42)\nGRID = {\n    "max_depth": [1, 2, 3],\n    "n_estimators": [50, 100],\n    "learning_rate": [0.1, 0.5],\n}\n\n\ndef make_adaboost(seed, params):\n    p = dict(params)\n    base_depth = p.pop("max_depth", 2)\n    base = DecisionTreeClassifier(max_depth=base_depth, random_state=seed)\n    return AdaBoostClassifier(estimator=base, random_state=seed, **p)\n\n\ndef run(output):\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(f"Choose an empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    dev = load_development()\n    tuning, evaluation = [], []\n    best = None\n    for fold, train, val in development_folds(dev):\n        prep = model_pipeline("adaboost", make_adaboost(SEEDS[0], {"max_depth": 2}))[0]\n        X = prep.fit_transform(train[FEATURES]).astype(np.float32)\n        V = prep.transform(val[FEATURES]).astype(np.float32)\n        y, target = train[TARGET], val[TARGET]\n        if fold <= 2:\n            configs, seeds = ParameterGrid(GRID), SEEDS[:1]\n        else:\n            if best is None:\n                raise ValueError("The first two tuning folds must precede evaluation.")\n            configs, seeds = [best], SEEDS\n        for params in configs:\n            for seed in seeds:\n                start = perf_counter()\n                model = make_adaboost(seed, params).fit(X, y)\n                prob = model.predict_proba(V)[:, 1]\n                row = dict(\n                    fold=fold,\n                    seed=seed,\n                    **params,\n                    train_rows=len(y),\n                    validation_rows=len(target),\n                    late_rate=target.mean(),\n                    ap=average_precision_score(target, prob),\n                    roc_auc=roc_auc_score(target, prob),\n                    seconds=perf_counter() - start,\n                )\n                (tuning if fold <= 2 else evaluation).append(row)\n                if fold > 2:\n                    val[[TARGET]].assign(probability=prob).to_parquet(\n                        output / f"predictions_{fold}_{seed}.parquet"\n                    )\n        print(\n            f"Finished fold {fold}: {len(y):,} training, {len(target):,} validation.",\n            flush=True,\n        )\n        if fold == 2:\n            table = pd.DataFrame(tuning)\n            table.to_csv(output / "tuning.csv", index=False)\n            ranking = table.groupby(list(GRID), as_index=False)[\n                ["ap", "roc_auc"]\n            ].mean()\n            winner = ranking.sort_values(\n                ["ap", "roc_auc", "n_estimators"], ascending=[False, False, True]\n            ).iloc[0]\n            best = {\n                "max_depth": int(winner.max_depth),\n                "n_estimators": int(winner.n_estimators),\n                "learning_rate": float(winner.learning_rate),\n            }\n            print("Selected", best, flush=True)\n        del X, V\n    results = pd.DataFrame(evaluation)\n    results.to_csv(output / "evaluation.csv", index=False)\n    means = results.groupby("fold")[["ap", "roc_auc", "late_rate"]].mean()\n    print(means.to_string(), flush=True)\n    print("Mean across evaluation folds", means.mean().to_dict(), flush=True)\n    final = model_pipeline("adaboost", make_adaboost(SEEDS[0], best)).fit(\n        dev[FEATURES], dev[TARGET]\n    )\n    joblib.dump(final, output / "adaboost.joblib", compress=3)\n    (output / "settings.json").write_text(\n        json.dumps(\n            {\n                "params": best,\n                "refit_seed": SEEDS[0],\n                "features": FEATURES,\n                "development_rows": len(dev),\n                "mean_validation": means.mean().to_dict(),\n            },\n            indent=2,\n        )\n        + "\\n"\n    )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, default=Path("runs/adaboost"))\n    run(parser.parse_args().out)\n', 'logistic_regression.py': '"""Tune T-2h logistic regression on folds 1-2, evaluate on 3-4 and save a development refit."""\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import ParameterGrid\n\nfrom preprocessing import (\n    FEATURES,\n    TARGET,\n    development_folds,\n    load_development,\n    model_pipeline,\n)\n\nSEEDS = (11, 23, 42)\nGRID = {\n    "C": [0.01, 0.1, 1.0, 10.0],\n}\n\n\ndef make_logreg(seed, params):\n    return LogisticRegression(\n        random_state=seed, max_iter=1000, solver="lbfgs", **params\n    )\n\n\ndef run(output):\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(f"Choose an empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    dev = load_development()\n    tuning, evaluation = [], []\n    best = None\n    for fold, train, val in development_folds(dev):\n        prep = model_pipeline(\n            "logistic_regression", make_logreg(SEEDS[0], {})\n        )[0]\n        X = prep.fit_transform(train[FEATURES]).astype(np.float32)\n        V = prep.transform(val[FEATURES]).astype(np.float32)\n        y, target = train[TARGET], val[TARGET]\n        if fold <= 2:\n            configs, seeds = ParameterGrid(GRID), SEEDS[:1]\n        else:\n            if best is None:\n                raise ValueError("The first two tuning folds must precede evaluation.")\n            configs, seeds = [best], SEEDS\n        for params in configs:\n            for seed in seeds:\n                start = perf_counter()\n                model = make_logreg(seed, params).fit(X, y)\n                prob = model.predict_proba(V)[:, 1]\n                row = dict(\n                    fold=fold,\n                    seed=seed,\n                    **params,\n                    train_rows=len(y),\n                    validation_rows=len(target),\n                    late_rate=target.mean(),\n                    ap=average_precision_score(target, prob),\n                    roc_auc=roc_auc_score(target, prob),\n                    seconds=perf_counter() - start,\n                )\n                (tuning if fold <= 2 else evaluation).append(row)\n                if fold > 2:\n                    val[[TARGET]].assign(probability=prob).to_parquet(\n                        output / f"predictions_{fold}_{seed}.parquet"\n                    )\n        print(\n            f"Finished fold {fold}: {len(y):,} training, {len(target):,} validation.",\n            flush=True,\n        )\n        if fold == 2:\n            table = pd.DataFrame(tuning)\n            table.to_csv(output / "tuning.csv", index=False)\n            ranking = table.groupby(list(GRID), as_index=False)[\n                ["ap", "roc_auc"]\n            ].mean()\n            winner = ranking.sort_values(\n                ["ap", "roc_auc", "C"], ascending=[False, False, True]\n            ).iloc[0]\n            best = {"C": float(winner.C)}\n            print("Selected", best, flush=True)\n        del X, V\n    results = pd.DataFrame(evaluation)\n    results.to_csv(output / "evaluation.csv", index=False)\n    means = results.groupby("fold")[["ap", "roc_auc", "late_rate"]].mean()\n    print(means.to_string(), flush=True)\n    print("Mean across evaluation folds", means.mean().to_dict(), flush=True)\n    final = model_pipeline("logistic_regression", make_logreg(SEEDS[0], best)).fit(\n        dev[FEATURES], dev[TARGET]\n    )\n    joblib.dump(final, output / "logistic_regression.joblib", compress=3)\n    (output / "settings.json").write_text(\n        json.dumps(\n            {\n                "params": best,\n                "refit_seed": SEEDS[0],\n                "features": FEATURES,\n                "development_rows": len(dev),\n                "mean_validation": means.mean().to_dict(),\n            },\n            indent=2,\n        )\n        + "\\n"\n    )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, default=Path("runs/logistic_regression"))\n    run(parser.parse_args().out)\n', 'pca_logistic_regression.py': '"""Tune T-2h PCA logistic regression on folds 1-2, evaluate on 3-4 and save a development refit."""\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom time import perf_counter\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.decomposition import PCA\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.model_selection import ParameterGrid\n\nfrom preprocessing import (\n    FEATURES,\n    TARGET,\n    development_folds,\n    load_development,\n    model_pipeline,\n)\n\nSEEDS = (11, 23, 42)\nGRID = {\n    "C": [0.01, 0.1, 1.0, 10.0],\n}\n\n\ndef make_logreg(seed, params):\n    return LogisticRegression(\n        random_state=seed, max_iter=1000, solver="lbfgs", **params\n    )\n\n\ndef run(output):\n    if output.exists() and any(output.iterdir()):\n        raise ValueError(f"Choose an empty output directory: {output}")\n    output.mkdir(parents=True, exist_ok=True)\n    dev = load_development()\n    tuning, evaluation = [], []\n    best = None\n    for fold, train, val in development_folds(dev):\n        prep = model_pipeline(\n            "pca_logistic_regression", make_logreg(SEEDS[0], {})\n        )[0]\n        X_scaled = prep.fit_transform(train[FEATURES]).astype(np.float32)\n        pca = PCA(n_components=0.95, svd_solver="full")\n        X = pca.fit_transform(X_scaled)\n        V_scaled = prep.transform(val[FEATURES]).astype(np.float32)\n        V = pca.transform(V_scaled)\n        n_components = pca.n_components_\n        explained_var = float(pca.explained_variance_ratio_.sum())\n        y, target = train[TARGET], val[TARGET]\n        if fold <= 2:\n            configs, seeds = ParameterGrid(GRID), SEEDS[:1]\n        else:\n            if best is None:\n                raise ValueError("The first two tuning folds must precede evaluation.")\n            configs, seeds = [best], SEEDS\n        for params in configs:\n            for seed in seeds:\n                start = perf_counter()\n                model = make_logreg(seed, params).fit(X, y)\n                prob = model.predict_proba(V)[:, 1]\n                row = dict(\n                    fold=fold,\n                    seed=seed,\n                    **params,\n                    n_components=n_components,\n                    explained_variance=explained_var,\n                    train_rows=len(y),\n                    validation_rows=len(target),\n                    late_rate=target.mean(),\n                    ap=average_precision_score(target, prob),\n                    roc_auc=roc_auc_score(target, prob),\n                    seconds=perf_counter() - start,\n                )\n                (tuning if fold <= 2 else evaluation).append(row)\n                if fold > 2:\n                    val[[TARGET]].assign(probability=prob).to_parquet(\n                        output / f"predictions_{fold}_{seed}.parquet"\n                    )\n        print(\n            f"Finished fold {fold}: {len(y):,} training, {len(target):,} validation, "\n            f"{n_components} PCA components ({explained_var:.4f} variance).",\n            flush=True,\n        )\n        if fold == 2:\n            table = pd.DataFrame(tuning)\n            table.to_csv(output / "tuning.csv", index=False)\n            ranking = table.groupby(list(GRID), as_index=False)[\n                ["ap", "roc_auc"]\n            ].mean()\n            winner = ranking.sort_values(\n                ["ap", "roc_auc", "C"], ascending=[False, False, True]\n            ).iloc[0]\n            best = {"C": float(winner.C)}\n            print("Selected", best, flush=True)\n        del X, V, X_scaled, V_scaled\n    results = pd.DataFrame(evaluation)\n    results.to_csv(output / "evaluation.csv", index=False)\n    means = results.groupby("fold")[["ap", "roc_auc", "late_rate"]].mean()\n    print(means.to_string(), flush=True)\n    print("Mean across evaluation folds", means.mean().to_dict(), flush=True)\n    final = model_pipeline(\n        "pca_logistic_regression", make_logreg(SEEDS[0], best)\n    ).fit(dev[FEATURES], dev[TARGET])\n    joblib.dump(final, output / "pca_logistic_regression.joblib", compress=3)\n    (output / "settings.json").write_text(\n        json.dumps(\n            {\n                "params": best,\n                "refit_seed": SEEDS[0],\n                "features": FEATURES,\n                "development_rows": len(dev),\n                "mean_validation": means.mean().to_dict(),\n            },\n            indent=2,\n        )\n        + "\\n"\n    )\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--out", type=Path, default=Path("runs/pca_logistic_regression"))\n    run(parser.parse_args().out)\n', 'build_features.py': '"""Create checkpoint-specific, chronological flight-delay modelling CSVs."""\n\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport numpy as np\nimport pandas as pd\nfrom pandas.tseries.holiday import USFederalHolidayCalendar\n\nfrom pipeline import iem\n\nROOT = Path(__file__).resolve().parent\nOUT = ROOT / "final_csv"\nWX = [\n    "skyc1",\n    "tmpf",\n    "dwpf",\n    "drct",\n    "sknt",\n    "gust",\n    "vsby",\n    "p01i",\n    "skyl1",\n    "obs_age_min",\n    "ts",\n    "fg",\n    "ra",\n    "sn",\n    "fz",\n    "br_hz",\n    "gust_missing",\n    "skyl1_missing",\n]\nSHARED = [\n    "month",\n    "dow",\n    "dep_hour",\n    "arr_hour",\n    "CRSElapsedTime",\n    "Distance",\n    "is_holiday",\n    "Origin",\n    "Dest",\n    "Reporting_Airline",\n]\nHISTORY = [\n    "n_departed_60min",\n    "prior_mean_delay_60min",\n    "late_rate_60min",\n    "n_scheduled_next_60min",\n    "has_history",\n]\nMETA = ["row_id", "FlightDate", "scheduled_dep_utc", "checkpoint_utc", "wx_obs_time"]\n\n\ndef activity(frame, checkpoint, allowed, protected, actual):\n    result = pd.DataFrame(index=frame.index, columns=HISTORY, dtype=float)\n    for origin, group in frame.groupby("Origin"):\n        rows = group.index\n        t = checkpoint.loc[rows]\n        departures = actual.loc[rows[allowed.loc[rows]]].sort_values()\n        delay = frame.DepDelay.loc[departures.index]\n        start = departures.searchsorted(t - pd.Timedelta(hours=1), side="right")\n        end = departures.searchsorted(t, side="right")\n        own = (\n            allowed.loc[rows]\n            & actual.loc[rows].gt(t - pd.Timedelta(hours=1))\n            & actual.loc[rows].le(t)\n        )\n        count = end - start - own.astype(int)\n        result.loc[rows, HISTORY[0]] = count\n        result.loc[rows, "has_history"] = count.gt(0).astype(int)\n        for col, values, own_values in [\n            (HISTORY[1], delay, group.DepDelay),\n            (HISTORY[2], delay.ge(15), group.DepDelay.ge(15)),\n        ]:\n            total = np.r_[0, values.to_numpy().cumsum()]\n            result.loc[rows, col] = (\n                total[end] - total[start] - own_values.astype(float).where(own, 0)\n            ) / count.replace(0, np.nan)\n        schedule = group.scheduled_dep_utc.sort_values()\n        result.loc[rows, HISTORY[3]] = schedule.searchsorted(\n            t + pd.Timedelta(hours=1), side="right"\n        ) - schedule.searchsorted(t, side="right")\n        # Withhold outcome-history fields when the window overlaps a sealed local date.\n        # A partial count would incorrectly imply that held-out events were observed.\n        blocked = protected.loc[rows]\n        if blocked.any():\n            zone = {\n                "ATL": "America/New_York",\n                "ORD": "America/Chicago",\n                "LAX": "America/Los_Angeles",\n            }[origin]\n            local = t.dt.tz_convert(zone)\n            sealed_dates = set(group.FlightDate[blocked])\n            unknown = local.dt.strftime("%Y-%m-%d").isin(sealed_dates) | (\n                (local - pd.Timedelta(hours=1))\n                .dt.strftime("%Y-%m-%d")\n                .isin(sealed_dates)\n            )\n            result.loc[rows[unknown], [HISTORY[i] for i in (0, 1, 2, 4)]] = np.nan\n    return result\n\n\ndef main():\n    frame = pd.read_parquet(iem.OUTPUT_PARQUET)\n    dates = frame.FlightDate\n    schedule = frame.scheduled_dep_utc\n    assert schedule.notna().all()\n    actual = schedule + pd.to_timedelta(frame.DepDelay, unit="min")\n    eligible = frame.Cancelled.eq(0) & frame.DepDelay.notna() & frame.DepDel15.notna()\n    assert (\n        frame.loc[eligible, "DepDelay"].ge(15).eq(frame.loc[eligible, "DepDel15"]).all()\n    )\n    old_test = frame.Origin.eq("ATL") & (\n        dates.between("2025-01-27", "2025-01-31")\n        | dates.between("2025-05-13", "2025-05-29")\n    )\n    test = dates.between("2025-12-01", "2025-12-31")\n    protected = old_test\n    allowed = eligible & ~protected\n    validation = eligible & dates.between("2025-09-01", "2025-11-28")\n    train = allowed & dates.between("2025-01-01", "2025-08-30")\n    # Label must have become observable before the earliest validation prediction.\n    train &= actual.lt((schedule[validation] - pd.Timedelta(hours=6)).min())\n    first_test_prediction = (schedule[test & eligible] - pd.Timedelta(hours=6)).min()\n    validation &= actual.lt(first_test_prediction)\n    masks = {"train": train, "validation": validation, "test": test & eligible}\n    assert not (train & validation | train & test | validation & test).any()\n    assert set(frame.Origin.unique()) == {"ATL", "ORD", "LAX"}\n    assert schedule[train].max() < (schedule[validation] - pd.Timedelta(hours=6)).min()\n    assert schedule[validation].max() < first_test_prediction\n    assert dates[test & eligible].min() == "2025-12-01"\n    assert dates[test & eligible].max() == "2025-12-31"\n    day = pd.to_datetime(dates)\n    shared = frame[\n        ["CRSElapsedTime", "Distance", "Origin", "Dest", "Reporting_Airline"]\n    ].copy()\n    shared["month"], shared["dow"] = day.dt.month, day.dt.dayofweek\n    shared["dep_hour"] = (frame.CRSDepTime // 100) % 24\n    shared["arr_hour"] = (frame.CRSArrTime // 100) % 24\n    holidays = USFederalHolidayCalendar().holidays(day.min(), day.max())\n    shared["is_holiday"] = day.isin(holidays).astype(int)\n    OUT.mkdir(exist_ok=True)\n    with TemporaryDirectory(dir=OUT) as temp:\n        staged = Path(temp)\n        for hours in (2, 6):\n            suffix = f"_T{hours}h"\n            t = schedule - pd.Timedelta(hours=hours)\n            obs = frame["wx_obs_time" + suffix]\n            age = (t - obs).dt.total_seconds() / 60\n            assert (\n                age.equals(frame["wx_obs_age_min" + suffix])\n                and age.dropna().between(0, 180).all()\n            )\n            wx = frame[["wx_" + s + suffix for s in WX]].copy()\n            wx["codes_known" + suffix] = frame["wx_ts" + suffix].notna().astype(int)\n            wx["wx_unmatched" + suffix] = obs.isna().astype(int)\n            history = activity(frame, t, allowed, protected, actual)\n            # Independent event-window audit on development rows, including empty windows.\n            for i in dates[train | validation].sample(300, random_state=42).index:\n                same = frame.Origin.eq(frame.at[i, "Origin"])\n                past = (\n                    allowed\n                    & same\n                    & actual.gt(t[i] - pd.Timedelta(hours=1))\n                    & actual.le(t[i])\n                )\n                past.at[i] = False\n                delays = frame.DepDelay[past]\n                demand = (\n                    same & schedule.gt(t[i]) & schedule.le(t[i] + pd.Timedelta(hours=1))\n                ).sum()\n                assert history.at[i, HISTORY[3]] == demand\n                if pd.notna(history.at[i, HISTORY[0]]):\n                    expected = [\n                        len(delays),\n                        delays.mean(),\n                        delays.ge(15).mean(),\n                        int(len(delays) > 0),\n                    ]\n                    np.testing.assert_allclose(\n                        history.loc[i, [HISTORY[k] for k in (0, 1, 2, 4)]],\n                        expected,\n                        equal_nan=True,\n                    )\n            inputs = pd.concat([shared[SHARED], wx, history.add_suffix(suffix)], axis=1)\n            assert inputs.shape[1] == 35\n            meta = pd.DataFrame(\n                {\n                    "row_id": frame.index,\n                    "FlightDate": dates,\n                    "scheduled_dep_utc": schedule,\n                    "checkpoint_utc": t,\n                    "wx_obs_time": obs,\n                }\n            )\n            table = pd.concat([meta, inputs, frame.DepDel15.rename("LATE")], axis=1)\n            for name, mask in masks.items():\n                part = table.loc[mask].sort_values(["checkpoint_utc", "row_id"])\n                part.to_csv(\n                    staged / f"{name}{suffix}.csv", index=False, chunksize=30000\n                )\n                print(\n                    name,\n                    suffix,\n                    len(part),\n                    part.FlightDate.min(),\n                    part.FlightDate.max(),\n                    flush=True,\n                )\n        for path in staged.glob("*.csv"):\n            path.replace(OUT / path.name)\n\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/bts.py': '"""Build a 2025 departure cohort and a separate history of flights into its airports."""\n\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import UTC, date, datetime, timedelta\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport pandas as pd\nimport requests\n\nYEAR = 2025\nAIRPORTS = {"ATL", "ORD", "LAX"}\nROOT = Path(__file__).resolve().parents[1] / "fresh_2025"\nBASE = (\n    "https://transtats.bts.gov/PREZIP/"\n    "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"\n)\nFIELDS = [\n    "FlightDate",\n    "Reporting_Airline",\n    "Tail_Number",\n    "Flight_Number_Reporting_Airline",\n    "Origin",\n    "Dest",\n    "CRSDepTime",\n    "DepTime",\n    "DepDelay",\n    "DepDel15",\n    "CRSArrTime",\n    "ArrTime",\n    "ArrDelay",\n    "Cancelled",\n    "Diverted",\n    "CRSElapsedTime",\n    "Distance",\n]\n\n\ndef download_month(year, month):\n    url = BASE.format(year=year, month=month)\n    path = ROOT / "bts_archives" / f"{year}-{month:02}.zip"\n    path.parent.mkdir(exist_ok=True)\n    if path.exists():\n        return path\n    partial = path.with_suffix(".part")\n    for attempt in range(3):\n        try:\n            with requests.get(url, stream=True, timeout=(20, 180)) as response:\n                response.raise_for_status()\n                with partial.open("wb") as out:\n                    for chunk in response.iter_content(1024 * 1024):\n                        out.write(chunk)\n            with zipfile.ZipFile(partial) as archive:\n                if archive.testzip() is not None:\n                    raise zipfile.BadZipFile("CRC failure")\n            partial.replace(path)\n            with (ROOT / "bts_download_log.txt").open("a") as log:\n                print(\n                    datetime.now(UTC).isoformat(),\n                    url,\n                    path.stat().st_size,\n                    file=log,\n                )\n            return path\n        except (requests.RequestException, zipfile.BadZipFile):\n            if attempt == 2:\n                raise\n            time.sleep(5 * 2**attempt)\n\n\ndef main():\n    ROOT.mkdir(exist_ok=True)\n    targets, inbound = [], []\n    # Two lookback days also cover US local-date differences for a 24-hour history.\n    earliest = (date(YEAR, 1, 1) - timedelta(days=2)).isoformat()\n    months = [(YEAR - 1, 12)] + [(YEAR, m) for m in range(1, 13)]\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        jobs = [pool.submit(download_month, year, month) for year, month in months]\n        for (year, month), job in zip(months, jobs):\n            with zipfile.ZipFile(job.result()) as archive:\n                (name,) = [n for n in archive.namelist() if n.lower().endswith(".csv")]\n                with archive.open(name) as stream:\n                    for frame in pd.read_csv(\n                        stream,\n                        usecols=FIELDS,\n                        chunksize=100_000,\n                        dtype={"Tail_Number": "string"},\n                    ):\n                        active = frame.FlightDate.ge(earliest)\n                        targets.append(\n                            frame.loc[\n                                active & frame.Origin.isin(AIRPORTS), FIELDS\n                            ].copy()\n                        )\n                        inbound.append(\n                            frame.loc[active & frame.Dest.isin(AIRPORTS), FIELDS].copy()\n                        )\n            print(f"Read {year}-{month:02d}", flush=True)\n    with TemporaryDirectory(dir=ROOT) as temp:\n        for parts, name in [\n            (targets, "flights_raw.parquet"),\n            (inbound, "inbound_history.parquet"),\n        ]:\n            destination = Path(temp) / name\n            data = pd.concat(parts, ignore_index=True)\n            data.to_parquet(destination, index=False)\n            destination.replace(ROOT / name)\n            print(name, len(data))\n\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/iem.py': '"""Attach IEM airport observations to flights as of each prediction checkpoint."""\n\nimport time\nfrom datetime import UTC, datetime, timedelta\nfrom email.utils import parsedate_to_datetime\nfrom io import BytesIO\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport airportsdata\nimport pandas as pd\nimport requests\n\nCHECKPOINT_HOURS = [2, 6]\nMAX_OBSERVATION_AGE = pd.Timedelta(hours=3)\nREQUEST_PAUSE_SECONDS = 10\nREQUEST_ATTEMPTS = 3\n\nROOT = Path(__file__).resolve().parents[1] / "fresh_2025"\nFLIGHTS = ROOT / "flights_raw.parquet"\nWEATHER_CACHE = ROOT / "weather_raw"\nWEATHER_PARQUET = ROOT / "weather_raw.parquet"\nDOWNLOAD_LOG = ROOT / "download_log.txt"\nOUTPUT_CSV = ROOT / "output_with_weather.csv"\nOUTPUT_PARQUET = ROOT / "output_with_weather.parquet"\nIEM_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"\nIEM_FIELDS = (\n    "tmpf", "dwpf", "drct", "sknt", "gust", "vsby", "p01i", "skyc1", "skyl1", "wxcodes",\n)\nSCHEDULE_FIELDS = ("FlightDate", "Origin", "CRSDepTime")\nOUTCOME_FIELDS = {\n    "DepTime", "DepDelay", "DepDel15", "ArrTime", "ArrDelay", "Cancelled", "Diverted",\n}\nUTC_DTYPE = "datetime64[ns, UTC]"\n\n\nclass DownloadError(RuntimeError):\n    """IEM failed to return a valid CSV for the requested interval."""\n\n\ndef log_download(message):\n    timestamp = datetime.now(UTC).isoformat(timespec="seconds")\n    with DOWNLOAD_LOG.open("a", encoding="utf-8") as log:\n        print(f"{timestamp} {message}", file=log)\n\n\ndef read_station_csv(source, origin):\n    raw = pd.read_csv(source, dtype="string", keep_default_na=False, comment="#")\n    columns = ("station", "valid", *IEM_FIELDS)\n    if not set(columns).issubset(raw.columns):\n        raise ValueError(f"{origin}: expected an IEM CSV with {\', \'.join(columns)}")\n    raw = raw.loc[:, list(columns)]\n    if not raw["station"].isin((origin, f"K{origin}")).all():\n        raise ValueError(f"{origin}: unexpected station IDs {raw[\'station\'].unique().tolist()}")\n    raw["valid"] = pd.to_datetime(raw["valid"], utc=True, errors="coerce").astype(UTC_DTYPE)\n    if raw["valid"].isna().any():\n        raise ValueError(f"{origin}: IEM CSV contains unparseable observation timestamps")\n    return raw\n\n\ndef retry_after_seconds(response):\n    header = response.headers.get("Retry-After") if response is not None else None\n    if not header:\n        return 0\n    if header.isdecimal():\n        return int(header)\n    try:\n        return max(0, (parsedate_to_datetime(header) - datetime.now(UTC)).total_seconds())\n    except (TypeError, ValueError):\n        return 0\n\n\nclass IEMDownloader:\n    def __init__(self, session):\n        self.session = session\n        self.next_request = 0\n\n    def fetch(self, origin, station, start, end):\n        params = [\n            ("station", station),\n            *(("data", field) for field in IEM_FIELDS),\n            ("year1", start.year), ("month1", start.month), ("day1", start.day),\n            ("year2", end.year), ("month2", end.month), ("day2", end.day),\n            ("tz", "Etc/UTC"), ("format", "onlycomma"), ("latlon", "no"),\n            ("missing", "M"), ("trace", "T"), ("report_type", 3), ("report_type", 4),\n        ]\n        url = requests.Request("GET", IEM_URL, params=params).prepare().url\n        for attempt in range(REQUEST_ATTEMPTS):\n            delay = self.next_request - time.monotonic()\n            if delay > 0:\n                time.sleep(delay)\n            response = None\n            log_download(f"GET attempt={attempt + 1} {url}")\n            try:\n                response = self.session.get(url, timeout=(15, 120))\n                response.raise_for_status()\n                raw = read_station_csv(BytesIO(response.content), origin)\n            except (requests.RequestException, ValueError) as error:\n                log_download(f"FAILED station={station} {type(error).__name__}: {error}")\n                if attempt == REQUEST_ATTEMPTS - 1:\n                    self.next_request = time.monotonic() + max(\n                        REQUEST_PAUSE_SECONDS, retry_after_seconds(response)\n                    )\n                    raise DownloadError(f"{station} {start} through {end} (exclusive)") from error\n                print(f"{station}: request attempt {attempt + 1} failed ({error}); retrying.")\n                self.next_request = time.monotonic() + max(\n                    REQUEST_PAUSE_SECONDS * 2**attempt, retry_after_seconds(response)\n                )\n            else:\n                self.next_request = time.monotonic() + REQUEST_PAUSE_SECONDS\n                log_download(f"OK rows={len(raw)} {response.url}")\n                return response.content, raw\n\n    def fetch_range(self, origin, station, start, end):\n        try:\n            return self.fetch(origin, station, start, end)\n        except DownloadError:\n            first_boundary = (pd.Timestamp(start) + pd.offsets.MonthBegin()).date()\n            if first_boundary >= end:\n                raise\n\n        print(f"{station}: full-range download failed; requesting calendar months.")\n        frames = []\n        while start < end:\n            boundary = min((pd.Timestamp(start) + pd.offsets.MonthBegin()).date(), end)\n            _, raw = self.fetch(origin, station, start, boundary)\n            frames.append(raw)\n            start = boundary\n        raw = pd.concat(frames, ignore_index=True)\n        content = raw.to_csv(index=False, date_format="%Y-%m-%d %H:%M").encode("utf-8")\n        return content, raw\n\n\ndef load_weather(stations, start, end):\n    """Reuse the station CSVs cached for the flight dataset\'s date window."""\n    cache_dir = WEATHER_CACHE / f"{start}_{end}"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    frames = []\n    with requests.Session() as session:\n        downloader = IEMDownloader(session)\n        for origin in stations:\n            cache = cache_dir / f"{origin}.csv"\n            try:\n                raw = read_station_csv(cache, origin)\n            except FileNotFoundError:\n                content, raw = downloader.fetch_range(origin, origin, start, end)\n                if raw.empty:\n                    print(f"{origin}: no observations; trying K{origin}.")\n                    content, raw = downloader.fetch_range(origin, f"K{origin}", start, end)\n                with TemporaryDirectory(dir=WEATHER_CACHE) as temp_dir:\n                    temporary = Path(temp_dir) / cache.name\n                    temporary.write_bytes(content)\n                    temporary.replace(cache)\n                print(f"{origin}: cached {len(raw):,} observations.")\n            else:\n                print(f"{origin}: loaded {len(raw):,} observations from cache.")\n            if raw.empty:\n                print(f"{origin}: no IEM observations available in the downloaded range.")\n            # Retain the IEM identifier and use the BTS airport code for the join.\n            frames.append(raw.assign(Origin=origin))\n\n    combined = pd.concat(frames, ignore_index=True)\n    in_range = combined["valid"].ge(pd.Timestamp(start, tz="UTC")) & combined["valid"].lt(\n        pd.Timestamp(end, tz="UTC")\n    )\n    return combined.loc[in_range].reset_index(drop=True)\n\n\ndef inspect_flights(flights):\n    print("Columns:", flights.columns.tolist())\n    print("Dtypes:")\n    print(flights.dtypes.to_string())\n    print("Head:")\n    print(flights.head().to_string(index=False))\n    print(f"Flight rows: {len(flights):,}")\n    if not set(SCHEDULE_FIELDS).issubset(flights.columns):\n        raise ValueError(f"Expected schedule columns {SCHEDULE_FIELDS}")\n\n    dates = pd.to_datetime(flights["FlightDate"], format="%Y-%m-%d", errors="coerce")\n    if dates.notna().sum() == 0:\n        raise ValueError("FlightDate contains no valid dates for the IEM request.")\n    first, last = dates.min().date(), dates.max().date()\n    print(f"FlightDate range: {first} through {last}")\n    stations = sorted(flights["Origin"].dropna().unique())\n    if not stations:\n        raise ValueError("Origin contains no airport codes for the IEM request.")\n    if any(len(code) != 3 or not code.isascii() or not code.isalnum() for code in stations):\n        raise ValueError(f"Expected three-character IATA origin codes; found {stations}")\n    print("Origin stations:", ", ".join(stations))\n    # IEM excludes its end date. Include the entire day after the last FlightDate.\n    start = first - timedelta(days=2)\n    end = last + timedelta(days=2)\n    print(f"IEM window (UTC): {start} through {end - timedelta(days=1)}, inclusive")\n    return stations, start, end\n\n\ndef scheduled_departures_utc(schedule):\n    # Only flight information known in advance can determine the checkpoint.\n    assert tuple(schedule.columns) == SCHEDULE_FIELDS\n    assert OUTCOME_FIELDS.isdisjoint(schedule.columns)\n\n    dates = pd.to_datetime(schedule["FlightDate"], format="%Y-%m-%d", errors="coerce")\n    hhmm = pd.to_numeric(schedule["CRSDepTime"], errors="coerce")\n    hours, minutes = hhmm // 100, hhmm % 100\n    valid = hhmm.ge(0) & hhmm.mod(1).eq(0) & (hhmm.lt(2400) | hhmm.eq(2400)) & minutes.lt(60)\n    # This treats e.g. 5 as 0005 and adds 24 hours for 2400.\n    local = dates + pd.to_timedelta((hours * 60 + minutes).where(valid), unit="min")\n\n    airports = airportsdata.load("IATA")\n    departure = pd.Series(pd.NaT, index=schedule.index, dtype=UTC_DTYPE, name="scheduled_dep_utc")\n    for origin, rows in schedule.groupby("Origin", sort=False).groups.items():\n        tz = airports.get(origin, {}).get("tz")\n        if not tz:\n            print(f"{origin}: no timezone available for {len(rows):,} rows.")\n            continue\n        # Use the first (DST) occurrence of the repeated fall-back hour at US airports.\n        # Advance a nonexistent spring-forward time to the first valid local time.\n        converted = local.loc[rows].dt.tz_localize(tz, ambiguous=True, nonexistent="shift_forward")\n        departure.loc[rows] = converted.dt.tz_convert("UTC").astype(UTC_DTYPE)\n    print(f"Scheduled departure UTC conversion failures: {departure.isna().sum():,} / {len(schedule):,}")\n    return departure\n\n\ndef build_weather_features(raw):\n    weather = raw.loc[:, ["Origin", "valid", *IEM_FIELDS]]\n    assert OUTCOME_FIELDS.isdisjoint(weather.columns)\n    features = weather.loc[:, ["Origin", "valid"]].copy()\n    missing = {"M": pd.NA, "": pd.NA}\n    for field in ("tmpf", "dwpf", "drct", "sknt", "gust", "vsby", "p01i", "skyl1"):\n        values = weather[field].replace(missing)\n        if field == "p01i":\n            # Keep the reported accumulation. IEM\'s numeric trace convention is 0.0001 inches.\n            values = values.replace("T", "0.0001")\n        features[f"wx_{field}"] = pd.to_numeric(values, errors="coerce").astype("float64")\n    features["wx_skyc1"] = weather["skyc1"].replace(missing)\n\n    for field in ("gust", "skyl1"):\n        values = features[f"wx_{field}"]\n        features[f"wx_{field}_missing"] = values.isna().astype("Int8")\n        features[f"wx_{field}_filled"] = values.fillna(0)\n\n    codes = weather["wxcodes"].replace(missing).str.upper()\n    # A missing code field leaves the indicators missing on that report.\n    for name, code in {"ts": "TS", "fg": "FG", "ra": "RA", "sn": "SN", "fz": "FZ", "br_hz": "BR|HZ"}.items():\n        # Match METAR\'s two-letter codes, including combinations such as -FZRA and +TSRA.\n        pattern = rf"(?:^|\\s)[+-]?(?:[A-Z]{{2}})*(?:{code})"\n        features[f"wx_{name}"] = codes.str.contains(pattern).astype("Int8")\n\n    return features\n\n\ndef attach_weather(flights, raw):\n    original_columns = flights.columns.tolist()\n    original_rows = len(flights)\n    schedule = flights.loc[:, list(SCHEDULE_FIELDS)]\n    departure = scheduled_departures_utc(schedule)\n    if "scheduled_dep_utc" in original_columns:\n        raise ValueError("Input already contains scheduled_dep_utc")\n\n    # merge_asof needs the time key globally sorted, even with a station grouping key.\n    left = pd.DataFrame({"Origin": schedule["Origin"].astype("string"), "departure": departure})\n    left = left.dropna(subset=["departure"]).sort_values("departure", kind="stable")\n    left = left.reset_index(names="flight_row")\n    # If IEM lists multiple reports at the same time, merge_asof uses the last listed report.\n    weather = build_weather_features(raw).sort_values("valid", kind="stable").reset_index(drop=True)\n    right = weather.loc[:, ["Origin", "valid"]].assign(weather_row=weather.index)\n    right["Origin"] = right["Origin"].astype("string")\n    values = weather.drop(columns=["Origin", "valid"])\n    flights["scheduled_dep_utc"] = departure\n\n    for hours in CHECKPOINT_HOURS:\n        left["T"] = left["departure"] - pd.Timedelta(hours=hours)\n        # Match only observations at or before T, at the same airport and within three hours.\n        # Join the weather row ID first to keep the time-key join small.\n        matches = pd.merge_asof(\n            left.loc[:, ["flight_row", "Origin", "T"]], right,\n            by="Origin", left_on="T", right_on="valid", direction="backward",\n            tolerance=MAX_OBSERVATION_AGE, allow_exact_matches=True,\n        )\n        ages = matches["T"] - matches["valid"]\n        if (ages.lt(pd.Timedelta(0)) | ages.gt(MAX_OBSERVATION_AGE)).any():\n            raise AssertionError(f"T-{hours}h: observation outside the allowed time window")\n\n        # Restore CSV row order, including rows with invalid schedules and unmatched weather.\n        matched = matches.set_index("flight_row").reindex(flights.index)\n        attached = values.reindex(matched["weather_row"].to_numpy()).set_axis(flights.index)\n        attached["wx_obs_time"] = matched["valid"]\n        attached["wx_obs_age_min"] = (matched["T"] - matched["valid"]).dt.total_seconds() / 60\n        attached = attached.add_suffix(f"_T{hours}h")\n        if set(attached.columns) & set(flights.columns):\n            raise ValueError(f"T-{hours}h: output feature names overlap existing columns")\n        for column in attached:\n            flights[column] = attached[column]\n\n        age_column = f"wx_obs_age_min_T{hours}h"\n        sample = flights.loc[:, ["Origin", "scheduled_dep_utc", f"wx_obs_time_T{hours}h", age_column]].head().copy()\n        sample.insert(2, "T", matched["T"].head())\n        print(f"\\nT-{hours}h timestamp audit sample:")\n        print(sample.to_string(index=False))\n        print(f"T-{hours}h flights with weather: {matched[\'valid\'].notna().sum():,} / {original_rows:,}")\n        print(f"T-{hours}h median weather age: {attached[age_column].median():.1f} min")\n        print(f"T-{hours}h missing by weather column:")\n        print(attached.isna().mean().mul(100).to_string(float_format=lambda value: f"{value:.3f}%"))\n\n    assert len(flights) == original_rows\n    assert flights.columns[:len(original_columns)].tolist() == original_columns\n    print(f"\\nFlight rows before / after: {original_rows:,} / {len(flights):,}")\n    print(f"New columns: {len(flights.columns) - len(original_columns)}")\n    return flights\n\n\ndef write_outputs(raw, flights):\n    # Stage the files so a failed write cannot leave a partially written CSV or Parquet.\n    with TemporaryDirectory(prefix=".weather-", dir=ROOT) as temp_dir:\n        staging = Path(temp_dir)\n        raw.to_parquet(staging / WEATHER_PARQUET.name, engine="pyarrow", index=False)\n        flights.to_parquet(staging / OUTPUT_PARQUET.name, engine="pyarrow", index=False)\n        flights.to_csv(\n            staging / OUTPUT_CSV.name, index=False, chunksize=50_000,\n            date_format="%Y-%m-%dT%H:%M:%SZ",\n        )\n        for output in (WEATHER_PARQUET, OUTPUT_PARQUET, OUTPUT_CSV):\n            (staging / output.name).replace(output)\n            print(f"Wrote {output.name}")\n\n\ndef main():\n    flights = pd.read_parquet(FLIGHTS)\n    stations, start, end = inspect_flights(flights)\n    raw = load_weather(stations, start, end)\n    print(f"IEM observations in the requested range: {len(raw):,}")\n    attach_weather(flights, raw)\n    print("Writing CSV and Parquet outputs...")\n    write_outputs(raw, flights)\n\n\nif __name__ == "__main__":\n    main()\n', 'pipeline/__init__.py': ''}

for rel_path, content in FILES.items():
    p = Path(rel_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')
    print(f'Wrote {rel_path}')

print('Setup complete!')


In [ ]:
# Cell 2: Prepare dataset (uses existing final_csv/ or builds from BTS + IEM)
from pathlib import Path
import subprocess

train_csv = Path('final_csv/train_T2h.csv')
val_csv = Path('final_csv/validation_T2h.csv')

if train_csv.exists() and val_csv.exists():
    print('Found existing CSVs in final_csv/!')
else:
    print('CSVs not found. Building dataset from BTS + IEM (takes ~5-10 min)...')
    subprocess.run(['python', 'pipeline/bts.py'], check=True)
    subprocess.run(['python', 'pipeline/iem.py'], check=True)
    subprocess.run(['python', 'build_features.py'], check=True)

!python preprocessing.py


In [ ]:
# Cell 3: Decision Tree (~2 min)
!rm -rf runs/decision_tree
!python decision_tree.py --out runs/decision_tree


In [ ]:
# Cell 4: Logistic Regression (~2 min)
!rm -rf runs/logistic_regression
!python logistic_regression.py --out runs/logistic_regression


In [ ]:
# Cell 5: PCA + Logistic Regression (~3 min)
!rm -rf runs/pca_logistic_regression
!python pca_logistic_regression.py --out runs/pca_logistic_regression


In [ ]:
# Cell 6: Random Forest (~25-35 min)
!rm -rf runs/random_forest
!python random_forest.py --out runs/random_forest


In [ ]:
# Cell 7: AdaBoost (~30-45 min)
!rm -rf runs/adaboost
!python adaboost.py --out runs/adaboost


In [ ]:
# Cell 8: Print summary of all models & download zip
import pandas as pd
from pathlib import Path

models = [
    'decision_tree',
    'logistic_regression',
    'pca_logistic_regression',
    'random_forest',
    'adaboost',
]

rows = []
for m in models:
    eval_path = Path(f'runs/{m}/evaluation.csv')
    if eval_path.exists():
        df = pd.read_csv(eval_path)
        fold_means = df.groupby('fold')[['ap', 'roc_auc', 'late_rate']].mean()
        rows.append({
            'model': m,
            'ap_mean': fold_means['ap'].mean(),
            'ap_std': fold_means['ap'].std(),
            'roc_auc_mean': fold_means['roc_auc'].mean(),
            'roc_auc_std': fold_means['roc_auc'].std(),
        })
    else:
        print(f'Missing {eval_path}')

if rows:
    summary = pd.DataFrame(rows).sort_values('ap_mean', ascending=False)
    print(summary.to_string(index=False))

!zip -r person2_tuned_runs.zip runs/
try:
    from google.colab import files
    files.download('person2_tuned_runs.zip')
except ImportError:
    print('Saved person2_tuned_runs.zip in current directory.')
